#Librería

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DateType, TimestampType, IntegerType

In [0]:
catalog = spark.sql("""select current_catalog()""").first()[0]
schema = "control"

#Control Table - Raw Ingestion Config

In [0]:
schema_raw_ingestion_config = StructType([
    StructField("entity_name",          StringType(),  False),  
    StructField("api_endpoint",         StringType(),  True),
    StructField("auth_method",          StringType(),  True),
    StructField("secret_name",          StringType(),  True),
    StructField("pagination_type",      StringType(),  True),
    StructField("response_format",      StringType(),  True),
    StructField("raw_path",             StringType(),  True),
    StructField("data_root_key",        StringType(),  True),
    StructField("extraction_frequency", StringType(),  True),
    StructField("is_active",            BooleanType(), True),
])

df_vacio_ingestion = spark.createDataFrame([], schema_raw_ingestion_config)


(
    df_vacio_ingestion.write
    .format("delta")
    .mode("ignore")   # idempotente: si ya existe, no la toca
    .saveAsTable("workspace.control.raw_ingestion_config")
)

print("Tabla 'raw_ingestion_config' verificada/creada.")

In [0]:
df = spark.table("workspace.control.raw_ingestion_config")
df.display()

#Control Table  - API Parameters

In [0]:
schema_raw_api_parameters = StructType([
    StructField("entity_name",  StringType(), False),
    StructField("param_name",   StringType(), False),
    StructField("param_value",  StringType(), True),
    StructField("param_type",   StringType(), True),
])

df_vacio_params = spark.createDataFrame([], schema_raw_api_parameters)

(
    df_vacio_params.write
    .format("delta")
    .mode("ignore")
    .saveAsTable(f"{catalog}.{schema}.raw_api_parameters")
)

print("Tabla 'raw_api_parameters' verificada/creada.")

In [0]:
df = spark.table("workspace.control.raw_api_parameters")
df.display()


#Control Table - Bronze Load Config

In [0]:

from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, DateType, TimestampType
)

schema_bronze_load_config = StructType([
    StructField("entity_name",      StringType(),    False),  # llave de enlace con raw_ingestion_config
    StructField("target_table",     StringType(),    True),   # ej. workspace.bronze.products
    StructField("primary_key",      StringType(),    True),   # columna id de negocio (usada recién en Silver)
    StructField("load_mode",        StringType(),    True),   # 'full' o 'incremental'
    StructField("watermark_column", StringType(),    True),   # ej. 'meta.updatedAt' — NULL si load_mode='full'
    StructField("last_watermark",   TimestampType(), True),   # último valor de watermark procesado con éxito
    StructField("last_loaded_date", DateType(),      True),   # última fecha en que corrió exitosamente
    StructField("last_run_status", StringType(),    True),   # 'success', 'failed', 'skipped'
    StructField("is_active",        BooleanType(),   True),
])

df_vacio_bronze_config = spark.createDataFrame([], schema_bronze_load_config)

(
    df_vacio_bronze_config.write
    .format("delta")
    .mode("ignore")
    .saveAsTable("workspace.control.bronze_load_config")
)

print("Tabla 'bronze_load_config' verificada/creada.")

In [0]:
spark.table("workspace.control.bronze_load_config").display()

#Control table - Silver Transform Config

In [0]:

schema_silver_transform_config = StructType([
    StructField("entity_name",         StringType(),  False),
    StructField("source_table",        StringType(),  True),
    StructField("target_table",        StringType(),  True),
    StructField("primary_key",         StringType(),  True),
    StructField("dedup_order_column",  StringType(),  True),
    StructField("last_run_status",     StringType(),  True),
    StructField("is_active",           BooleanType(), True),
])

df_vacio_silver_config = spark.createDataFrame([], schema_silver_transform_config)

(
    df_vacio_silver_config.write
    .format("delta")
    .mode("ignore")
    .saveAsTable("workspace.control.silver_transform_config")
)

print("Tabla 'silver_transform_config' verificada/creada.")

#Gold 

In [0]:
schema_gold_config = StructType([
    StructField("entity_name",   StringType(),  False),   # 'dim_products', 'dim_users', 'fact_cart_items'
    StructField("source_table",  StringType(),  True),
    StructField("target_table",  StringType(),  True),
    StructField("last_run_status", StringType(), True),
    StructField("is_active",     BooleanType(), True),
])

df_vacio_gold = spark.createDataFrame([], schema_gold_config)

(
    df_vacio_gold.write
    .format("delta")
    .mode("ignore")
    .saveAsTable("workspace.control.gold_aggregation_config")
)

print("Tabla 'gold_aggregation_config' verificada/creada.")